In [62]:
import pandas as pd

df_model = pd.read_csv("../data/processed/amazon_reviews_model.csv")
df_model.head()


,user_id_enc,item_id_enc,rating_norm,timestamp
0,1456,15167,-1.239889,2022-07-18 22:58:37.948
1,1456,13689,-3.037525,2020-06-20 18:42:29.731
2,1456,10383,0.557747,2018-04-07 09:23:37.534
3,2248,5393,0.557747,2010-11-20 18:41:35.000
4,1892,12523,0.557747,2023-02-17 02:39:41.238


In [63]:
df_model.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20990 entries, 0 to 20989
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   user_id_enc  20990 non-null  int64  
 1   item_id_enc  20990 non-null  int64  
 2   rating_norm  20990 non-null  float64
 3   timestamp    20990 non-null  object 
dtypes: float64(1), int64(2), object(1)
memory usage: 656.1+ KB


In [64]:
df_model["timestamp"] = pd.to_datetime(df_model["timestamp"], errors="coerce")


In [65]:
df_model.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20990 entries, 0 to 20989
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   user_id_enc  20990 non-null  int64         
 1   item_id_enc  20990 non-null  int64         
 2   rating_norm  20990 non-null  float64       
 3   timestamp    20990 non-null  datetime64[ns]
dtypes: datetime64[ns](1), float64(1), int64(2)
memory usage: 656.1 KB


In [66]:
df_model["user_id_enc"] = df_model["user_id_enc"].astype(int)
df_model["item_id_enc"] = df_model["item_id_enc"].astype(int)


In [67]:
assert df_model["rating_norm"].notna().all()


In [68]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df_model,
    test_size=0.2,
    random_state=42
)


In [69]:
user_item_matrix = df_model.pivot_table(
    index="user_id_enc",
    columns="item_id_enc",
    values="rating_norm",
    fill_value=0
)


In [70]:
from sklearn.decomposition import TruncatedSVD

n_components = 30

svd = TruncatedSVD(
    n_components=n_components,
    random_state=42
)

user_factors = svd.fit_transform(user_item_matrix)
item_factors = svd.components_.T


In [71]:
def recommend_lsa_eval(user_id_enc, k=5):
    if user_id_enc not in user_item_train.index:
        return []

    user_idx = user_item_train.index.get_loc(user_id_enc)
    scores = predicted_ratings[user_idx]

    seen_items = set(
        train_df[train_df["user_id_enc"] == user_id_enc]["item_id_enc"]
    )

    ranked_items = [
        user_item_train.columns[i]
        for i in np.argsort(scores)[::-1]
        if user_item_train.columns[i] not in seen_items
    ]

    return ranked_items[:k]


In [72]:
explained_variance = svd.explained_variance_ratio_.sum()
explained_variance


np.float64(0.29123834051465475)

In [73]:
import numpy as np

predicted_ratings = np.dot(user_factors, item_factors.T)


In [74]:
def recommend_lsa(user_id_enc, n=5):
    user_index = user_item_matrix.index.get_loc(user_id_enc)
    
    user_scores = predicted_ratings[user_index]
    
    # produits déjà vus
    seen_items = set(
        df_model[df_model["user_id_enc"] == user_id_enc]["item_id_enc"]
    )
    
    # classement des items
    item_indices = [
        i for i in np.argsort(user_scores)[::-1]
        if user_item_matrix.columns[i] not in seen_items
    ]
    
    top_items = [
        user_item_matrix.columns[i]
        for i in item_indices[:n]
    ]
    
    return top_items


In [75]:
test_user_enc = df_model["user_id_enc"].iloc[0]
recommend_lsa(test_user_enc, n=5)


[np.int64(12591),
 np.int64(17796),
 np.int64(16399),
 np.int64(11230),
 np.int64(7725)]

In [76]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df_model,
    test_size=0.2,
    random_state=42
)


In [77]:
from sklearn.metrics import mean_squared_error
import numpy as np

# prédictions LSA pour le jeu de test
y_true = test_df["rating_norm"].values

# reconstruire la matrice user-item train → prédictions
# (on simplifie ici : tu peux expliquer le principe)
y_pred = []

for _, row in test_df.iterrows():
    u = row["user_id_enc"]
    i = row["item_id_enc"]
    y_pred.append(predicted_ratings[u, i])

rmse = np.sqrt(mean_squared_error(y_true, y_pred))
rmse


np.float64(0.8454323923026589)

In [78]:
def precision_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]
    return len(set(recommended_k) & set(relevant)) / k


In [79]:
K = 5
precisions = []

for user in test_df["user_id_enc"].unique():
    relevant_items = test_df[
        test_df["user_id_enc"] == user
    ]["item_id_enc"].tolist()
    
    recommended_items = recommend_lsa(user, n=K)
    
    if relevant_items:
        precisions.append(
            precision_at_k(recommended_items, relevant_items, K)
        )

np.mean(precisions)


np.float64(0.0)

In [80]:

test_user = df_model["user_id_enc"].iloc[0]
test_user in user_item_matrix.index


True

In [81]:

n_items_total = user_item_matrix.shape[1]
n_seen = df_model[df_model["user_id_enc"] == test_user]["item_id_enc"].nunique()
n_items_total, n_seen


(19892, 22)

In [82]:

df_model[df_model["user_id_enc"] == test_user].shape[0]


22

In [83]:
from sklearn.neighbors import NearestNeighbors

knn = NearestNeighbors(n_neighbors=11, metric="cosine")  # 10 voisins + soi-même
knn.fit(user_item_matrix)


,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",11
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [84]:
import numpy as np

def recommend_knn_user_based(
    user_id_enc: int,
    n_neighbors: int = 10,
    n_items: int = 5,
    min_neighbor_rating: float = 0.0,   # tu peux mettre 0.1 si tu veux filtrer un peu
):
   
    if user_id_enc not in user_item_matrix.index:
        return []

    user_vec = user_item_matrix.loc[user_id_enc].values.reshape(1, -1)

    
    distances, indices = knn.kneighbors(user_vec, n_neighbors=n_neighbors + 1)
    neighbor_user_ids = user_item_matrix.index[indices.flatten()].tolist()

    # on enlève l'utilisateur lui-même s'il est dans la liste
    neighbor_user_ids = [u for u in neighbor_user_ids if u != user_id_enc]

   
    seen_items = set(df_model[df_model["user_id_enc"] == user_id_enc]["item_id_enc"].unique())

    # --- 3) Agrégation : score = moyenne des ratings des voisins ---
    neighbor_df = df_model[df_model["user_id_enc"].isin(neighbor_user_ids)].copy()

    # fallback
    if neighbor_df.empty:
        return []

    scores = (
        neighbor_df
        .groupby("item_id_enc")["rating_norm"]
        .mean()
        .sort_values(ascending=False)
    )

    # --- 4) Filtrer : pas déjà vus + score minimal ---
    recs = [item for item, s in scores.items() if (item not in seen_items and s > min_neighbor_rating)]

    # --- 5) Si c’est vide, fallback : on autorise des items même si score == 0 (ou on retire min_neighbor_rating) ---
    if len(recs) == 0:
        recs = [item for item in scores.index.tolist() if item not in seen_items]

    return recs[:n_items]


In [85]:
test_user = df_model["user_id_enc"].iloc[0]
recommend_knn_user_based(test_user, n_neighbors=10, n_items=5)


[14, 13087, 13430, 13399, 13323]